# Engram: fresh-data test on LongMemEval

Set `STAGE` below. First run `pilot`: 30 development questions to measure speed and check the adapter. It does not touch the test questions. Benchmark data only.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Run > Run all. Keep the tab open.


In [ ]:
STAGE = "pilot"
COMMIT = "1b16ed3"
import os, subprocess, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK = "/kaggle/working"
os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
os.chdir(f"{WORK}/Engram")
subprocess.run(f"git checkout -q {COMMIT}; git clone -q --depth 1 https://github.com/snap-research/locomo.git", shell=True, check=True)
subprocess.run("pip install -q wordllama scikit-learn spacy pytest; python -m spacy download en_core_web_sm", shell=True, check=True, capture_output=True)
import wordllama, shutil
p = os.path.dirname(wordllama.__file__)
a, b = os.path.join(p, "tokenizers"), os.path.join(p, "tokenizer")
if os.path.isdir(a) and not os.path.exists(b): shutil.copytree(a, b)
print("tests:", subprocess.run("python -m pytest -q -p no:cacheprovider 2>&1 | tail -1", shell=True, capture_output=True, text=True).stdout.strip())
DATA = f"{WORK}/longmemeval_s_cleaned.json"
if not os.path.exists(DATA):
    subprocess.run(f"wget -q -O {DATA} https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json", shell=True, check=True)
print("data:", os.path.getsize(DATA) // 1000000, "MB")
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="4")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(5)
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
print("setup ready")


In [ ]:
import os, subprocess, sys, time
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(f"{WORK}/Engram")
LOG = f"{SAVE}/lme_run.log"
if subprocess.run(["pgrep", "-f", "lme_test.py"], capture_output=True).returncode != 0:
    subprocess.Popen([sys.executable, "-u", "kit_tools/lme_test.py", "--stage", STAGE, "--data", f"{WORK}/longmemeval_s_cleaned.json", "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--save", SAVE], stdout=open(LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    print(STAGE, "started")
for _ in range(2880):
    time.sleep(15)
    t = open(LOG).read().splitlines()
    print(chr(13) + " | ".join(t[-3:])[-150:], end="")
    if t and (t[-1].startswith("Done") or "STOPPED" in t[-1] or "Traceback" in chr(10).join(t[-30:])): break
print(); print(chr(10).join(open(LOG).read().splitlines()[-45:]))


When it says Done, send Claude these from the `engram_results` folder in the Output panel: `lme_pilot.json`, `lme_stats.json`, `lme_pilot_answers.jsonl`, `lme_facts.json` and `lme_run.log`.
